# Stage 1 (long run): θ_ref on a Colab GPU

**Before you start**
1. In Google Drive, create the folder `MyDrive/unlearn/` and upload `colab/unlearn_bundle.tar.gz` into it (60 MB).
2. Choose **Runtime → Change runtime type → T4 GPU** (an L4 or A100 is faster, if you have one).
3. Run the cells in order. Checkpoints and logs go to `MyDrive/unlearn/`. If Colab disconnects, reconnect, run cells 1–4 again, then run cell 6 again: it resumes from the last checkpoint (at most 500 steps are lost).

The final model is `MyDrive/unlearn/checkpoints/ref_long_s0.pt`. Download it together with the `results/` folder and put both back into the project. Claude will then verify the SHA-256 and make the file read-only.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/unlearn'
import os; os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True); os.makedirs(f'{DRIVE}/results', exist_ok=True)
print(os.listdir(DRIVE))

In [ ]:
# 3. Unpack code + data onto the fast local disk and verify the data is byte-identical to the Mac copy
!rm -rf /content/unlearn && mkdir -p /content/unlearn
!tar xzf /content/drive/MyDrive/unlearn/unlearn_bundle.tar.gz -C /content/unlearn
%cd /content/unlearn
!cd data/processed && sha256sum -c MANIFEST.sha256

In [ ]:
# 4. Dependencies (torch/numpy/matplotlib/pytest are preinstalled on Colab) + model tests
!pip -q install pyyaml
import torch; print(torch.__version__, torch.cuda.get_device_name(0), 'bf16:', torch.cuda.is_bf16_supported())
!python -m pytest -q tests/test_model.py

In [ ]:
# 5. Timing run: 110 steps, also exercises the GPU mixed-precision path. Prints the projected run time.
!python scripts/train.py --config configs/stage1_long.yaml --seed 0 --timing --results-dir {DRIVE}/results
import json
t = [json.loads(l) for l in open(f'{DRIVE}/results/timing.jsonl')][-1]
steps = 6000
minutes = steps / t['steps_per_sec_after_warmup'] / 60 + (steps // 500 + 1) * t['seconds_per_eval'] / 60
print(f"{t['steps_per_sec_after_warmup']} steps/s -> projected {minutes:.0f} min for {steps} steps")

In [ ]:
# 6. Train (re-run this cell after a disconnect: --resume continues from the last checkpoint on Drive)
!python scripts/train.py --config configs/stage1_long.yaml --seed 0 --resume \
    --ckpt-dir {DRIVE}/checkpoints --results-dir {DRIVE}/results 2>&1 | grep -v Warning

In [ ]:
# 7. Check outputs: download checkpoints/ref_long_s0.pt and the whole results/ folder
!ls -la {DRIVE}/checkpoints {DRIVE}/results
!sha256sum {DRIVE}/checkpoints/ref_long_s0.pt && cat {DRIVE}/results/ref_hashes.json